In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt 
import seaborn as sns 
import pandas as pd 
import plotly.express as px  
import os

In [4]:
print(os.listdir("results_extended"))

['paper_scores.csv', 'author_scores_delta0.csv', 'author_scores_delta05.csv', 'author_scores_delta1.csv', 'sensitivity_delta.csv']


In [8]:
# small cohort analysis
paper_scores_A = pd.read_csv("results_cohortA/paper_scores.csv")
paper_scores_B = pd.read_csv("results_cohortB/paper_scores.csv")

cohort_paper_scores = pd.concat([paper_scores_A, paper_scores_B], ignore_index=True)

In [18]:
# Modifies the DataFrame in place
cohort_paper_scores = cohort_paper_scores.round(3)

cohort_paper_scores.to_clipboard()

In [10]:
author_scores_A = pd.read_csv("results_cohortA/author_scores_delta05.csv")
author_scores_B = pd.read_csv("results_cohortB/author_scores_delta05.csv")

cohort_author_scores = pd.concat([author_scores_A, author_scores_B], ignore_index=True)

In [14]:
multi_paper_author_scores = cohort_author_scores[cohort_author_scores["n_papers"] >= 2]

In [19]:
multi_paper_author_scores = multi_paper_author_scores.round(3)
multi_paper_author_scores.to_clipboard()

In [3]:
os.chdir("results_extended")

In [4]:
paper_scores = pd.read_csv("paper_scores.csv")
paper_scores.head()

,pub_anon_id,n_authors,delta_gender,delta_country,delta_discipline
0,W1038137833,5,0.0,0.000000,0.280000
1,W1084973187,4,0.0,0.072917,0.187500
2,W1185112711,2,0.0,0.000000,0.250000
3,W1201583382,7,0.0,0.081633,0.204082
4,W1219459344,5,0.0,0.000000,0.320000


In [8]:
scores = pd.read_csv("author_scores_delta05.csv")

In [9]:
scores.head()

,author_anon_id,n_papers,delta_gender,delta_country,delta_discipline,beta_mean,commindex_scalar
0,A5047867173,82,0.0,0.218495,0.332544,1.105964,0.0
1,A5100376706,10,0.0,0.194396,0.255104,1.287500,0.0
2,A5014540719,492,0.0,0.090990,0.320140,1.140531,0.0
3,A5087568651,40,0.0,0.179352,0.352000,1.090625,0.0
4,A5020061120,1,0.0,0.124444,0.393333,1.500000,0.0


In [10]:
multi = scores[scores["n_papers"] >= 2]

In [11]:
import plotly.subplots as sp
import plotly.graph_objects as go

fig = sp.make_subplots(rows=1, cols=2, subplot_titles=("Zero vs positive", "Distribution when positive"))

zeros = (multi["delta_country"] == 0).sum()
nonzeros = (multi["delta_country"] > 0).sum()

fig.add_trace(go.Bar(x=["Zero", "Positive"], y=[zeros, nonzeros]), row=1, col=1)
fig.add_trace(go.Histogram(x=multi.loc[multi["delta_country"] > 0, "delta_country"], nbinsx=30), row=1, col=2)
fig.update_layout(title="Country diversity — extended cohort", showlegend=False)
fig.show()

In [12]:
# Which seeds do the zero-country authors come from?
zero_country_authors = multi[multi["delta_country"] == 0]
print(f"{len(zero_country_authors)} authors with δ_country = 0")

# Cross-reference against extended cohort to find their seed_author_anon_id
extended = pd.read_csv("../adapted_extended_cohort.csv")
seed_map = extended.groupby("author_anon_id")["seed_author_anon_id"].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None)

zero_country_authors_with_seed = zero_country_authors.merge(seed_map.rename("seed"), left_on="author_anon_id", right_index=True, how="left")
print("Zero-country authors by their primary seed:")
print(zero_country_authors_with_seed["seed"].value_counts())

849 authors with δ_country = 0
Zero-country authors by their primary seed:
seed
A77c21dacbd16    218
Aa6270b01971b    163
A38327606686f    118
Ab1e8b19bae0b    108
A1d51b8d97097     96
Af103ef0d80ce     60
A11460d5aa670     35
A6e2aff8d4f11     24
A60af1130add6     20
Afb5914645166      7
Name: count, dtype: int64


In [13]:
# For each seed, what fraction of their network has zero country diversity?
extended = pd.read_csv("../adapted_extended_cohort.csv")
seed_map = extended.groupby("author_anon_id")["seed_author_anon_id"].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None)
multi_with_seed = multi.merge(seed_map.rename("seed"), left_on="author_anon_id", right_index=True, how="left")

by_seed = multi_with_seed.groupby("seed").agg(
    total_multi=("delta_country", "count"),
    zero_country=("delta_country", lambda x: (x == 0).sum())
)
by_seed["fraction_zero"] = by_seed["zero_country"] / by_seed["total_multi"]
by_seed = by_seed.sort_values("fraction_zero", ascending=False)
print(by_seed)

               total_multi  zero_country  fraction_zero
seed                                                   
A77c21dacbd16          332           218       0.656627
Afb5914645166           11             7       0.636364
A11460d5aa670           63            35       0.555556
Af103ef0d80ce          112            60       0.535714
Aa6270b01971b          321           163       0.507788
A60af1130add6           46            20       0.434783
A38327606686f          319           118       0.369906
A1d51b8d97097          293            96       0.327645
Ab1e8b19bae0b          382           108       0.282723
A6e2aff8d4f11          154            24       0.155844


In [20]:
extended = pd.read_csv("../adapted_extended_cohort.csv")
seed_map = extended.groupby("author_anon_id")["seed_author_anon_id"].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None
)
multi_with_seed = multi.merge(seed_map.rename("seed"), left_on="author_anon_id", right_index=True, how="left")

# Discipline breakdown by seed
by_seed_disc = multi_with_seed.groupby("seed").agg(
    total=("delta_discipline", "count"),
    mean_disc=("delta_discipline", "mean"),
    median_disc=("delta_discipline", "median"),
).sort_values("mean_disc", ascending=False)
print(by_seed_disc.round(3))

               total  mean_disc  median_disc
seed                                        
Aa6270b01971b    321      0.507        0.486
Af103ef0d80ce    112      0.504        0.452
A60af1130add6     46      0.466        0.470
A6e2aff8d4f11    154      0.418        0.410
A38327606686f    319      0.401        0.399
A77c21dacbd16    332      0.397        0.293
A11460d5aa670     63      0.352        0.362
Afb5914645166     11      0.345        0.256
Ab1e8b19bae0b    382      0.327        0.318
A1d51b8d97097    293      0.282        0.251
